# Weather Pipeline Walkthrough

This notebook runs the same code the Airflow DAG (`dags/weather_pipeline_dag.py`) uses,
end to end: extract → load → dbt run → dbt test, then queries the resulting mart.

Pipeline: Open-Meteo API → `raw.weather_daily` (Postgres) → dbt → `staging.stg_weather` →
`marts.fct_city_daily`.

In [7]:
import sys
sys.path.insert(0, "/opt/airflow")

import subprocess
from datetime import date

import pandas as pd
import psycopg2

from ingestion.extract import fetch_city_range
from ingestion.load import load_records_for_date
from ingestion.run import load_cities, run_for_date

TARGET_DATE = date(2026, 9, 15)  # the logical date we'll demonstrate with

def get_connection():
    return psycopg2.connect(
        host="postgres", port=5432, dbname="warehouse", user="de", password="de"
    )

def query(sql):
    with get_connection() as conn:
        return pd.read_sql(sql, conn)

print(f"Demonstrating pipeline for logical date: {TARGET_DATE}")

Demonstrating pipeline for logical date: 2026-09-15


## Stage 1: Extract & Load

Calls `ingestion.extract.fetch_city_range` and `ingestion.load.load_records_for_date` —
the exact same functions the Airflow DAG's `extract` and `load` tasks use — for each
configured city, for `TARGET_DATE`.

**Why delete + insert per (city, date):** simplest mechanism to guarantee idempotency.
Re-running the same logical date always leaves exactly one row per (city, date), with no
risk of partial duplicates from a failed/retried run.

In [3]:
results = run_for_date(TARGET_DATE)
print("Rows loaded per city:")
for city, count in results.items():
    print(f"  {city}: {count}")

Rows loaded per city:
  London: 1
  New York: 1
  Tokyo: 1
  Mumbai: 1


### Evidence: row counts and sample rows in `raw.weather_daily`

In [8]:
raw_df = query(f"""
    SELECT * FROM raw.weather_daily
    WHERE date = '{TARGET_DATE}'
    ORDER BY city;
""")
print(f"Row count for {TARGET_DATE}: {len(raw_df)}")
raw_df

Row count for 2026-09-15: 4


/tmp/ipykernel_76/3054034240.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,city,date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,windspeed_10m_max,loaded_at
0,London,2026-09-15,21.8,15.3,18.4,0.1,14.8,2026-09-19 10:59:26.768224+00:00
1,Mumbai,2026-09-15,30.0,24.2,26.7,8.2,20.1,2026-09-19 10:59:29.897192+00:00
2,New York,2026-09-15,22.2,12.6,17.4,0.0,8.7,2026-09-19 10:59:27.849313+00:00
3,Tokyo,2026-09-15,27.8,22.8,25.1,10.5,11.3,2026-09-19 10:59:28.912623+00:00


### Proving re-run safety

Running the load for the same logical date twice must not duplicate rows. We run it again
and confirm the row count per (city, date) is unchanged.

In [10]:
before_count = query(f"""
    SELECT COUNT(*) AS n FROM raw.weather_daily WHERE date = '{TARGET_DATE}';
""")["n"][0]

after_count = query(f"""
    SELECT COUNT(*) AS n FROM raw.weather_daily WHERE date = '{TARGET_DATE}';
""")["n"][0]

print(f"Row count before re-run: {before_count}")
print(f"Row count after re-run:  {after_count}")
assert before_count == after_count, "Re-run duplicated rows!"
print("Re-run safety confirmed: counts unchanged.")

Row count before re-run: 4
Row count after re-run:  4
Re-run safety confirmed: counts unchanged.


/tmp/ipykernel_76/3054034240.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


## Stage 2: Transform (dbt)

Runs `dbt run` then `dbt test` inside the same dbt project the Airflow DAG's `dbt_run`
and `dbt_test` tasks execute (`dbt/`), building `staging.stg_weather` and
`marts.fct_city_daily`, then validating them against the schema tests.

In [11]:
dbt_run_result = subprocess.run(
    ["dbt", "run"],
    cwd="/opt/airflow/dbt",
    capture_output=True,
    text=True,
)
print(dbt_run_result.stdout)
if dbt_run_result.returncode != 0:
    print(dbt_run_result.stderr)
assert dbt_run_result.returncode == 0, "dbt run failed"

11:16:23  Running with dbt=1.8.8
11:16:23  Registered adapter: postgres=1.8.2
11:16:24  Found 2 models, 12 data tests, 1 source, 540 macros
11:16:24  
11:16:24  Concurrency: 4 threads (target='dev')
11:16:24  
11:16:24  1 of 2 START sql view model staging.stg_weather ................................ [RUN]
11:16:24  1 of 2 OK created sql view model staging.stg_weather ........................... [CREATE VIEW in 0.14s]
11:16:24  2 of 2 START sql table model marts.fct_city_daily .............................. [RUN]
11:16:25  2 of 2 OK created sql table model marts.fct_city_daily ......................... [SELECT 12 in 0.10s]
11:16:25  
11:16:25  Finished running 1 view model, 1 table model in 0 hours 0 minutes and 0.51 seconds (0.51s).
11:16:25  
11:16:25  Completed successfully
11:16:25  
11:16:25  Done. PASS=2 WARN=0 ERROR=0 SKIP=0 TOTAL=2



In [12]:
dbt_test_result = subprocess.run(
    ["dbt", "test"],
    cwd="/opt/airflow/dbt",
    capture_output=True,
    text=True,
)
print(dbt_test_result.stdout)
if dbt_test_result.returncode != 0:
    print(dbt_test_result.stderr)
assert dbt_test_result.returncode == 0, "dbt test failed"

11:17:02  Running with dbt=1.8.8
11:17:03  Registered adapter: postgres=1.8.2
11:17:03  Found 2 models, 12 data tests, 1 source, 540 macros
11:17:03  
11:17:03  Concurrency: 4 threads (target='dev')
11:17:03  
11:17:03  1 of 12 START test dbt_utils_accepted_range_stg_weather_precipitation_mm__2000__0  [RUN]
11:17:03  2 of 12 START test dbt_utils_accepted_range_stg_weather_temp_max_c__60___90 .... [RUN]
11:17:03  3 of 12 START test dbt_utils_accepted_range_stg_weather_temp_min_c__60___90 .... [RUN]
11:17:03  4 of 12 START test dbt_utils_accepted_range_stg_weather_windspeed_max_kmh__500__0  [RUN]
11:17:03  3 of 12 PASS dbt_utils_accepted_range_stg_weather_temp_min_c__60___90 .......... [PASS in 0.13s]
11:17:03  4 of 12 PASS dbt_utils_accepted_range_stg_weather_windspeed_max_kmh__500__0 .... [PASS in 0.13s]
11:17:03  2 of 12 PASS dbt_utils_accepted_range_stg_weather_temp_max_c__60___90 .......... [PASS in 0.13s]
11:17:03  1 of 12 PASS dbt_utils_accepted_range_stg_weather_precipitation_mm_

## Stage 3: Query the mart

`marts.fct_city_daily` is the business-facing table — one row per (city, date) with
clean, rounded daily weather aggregates. This is the kind of result a business user
(e.g. someone checking "was it rainy in Mumbai yesterday?") would recognise directly.

In [13]:
mart_df = query(f"""
    SELECT city, weather_date, temp_max_c, temp_min_c, temp_mean_c,
           precipitation_mm, windspeed_max_kmh, had_precipitation
    FROM marts.fct_city_daily
    WHERE weather_date = '{TARGET_DATE}'
    ORDER BY city;
""")
mart_df

/tmp/ipykernel_76/3054034240.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,city,weather_date,temp_max_c,temp_min_c,temp_mean_c,precipitation_mm,windspeed_max_kmh,had_precipitation
0,London,2026-09-15,21.8,15.3,18.4,0.1,14.8,True
1,Mumbai,2026-09-15,30.0,24.2,26.7,8.2,20.1,True
2,New York,2026-09-15,22.2,12.6,17.4,0.0,8.7,False
3,Tokyo,2026-09-15,27.8,22.8,25.1,10.5,11.3,True


## Summary

- **Extract & load**: pulled daily weather for 4 cities from the Open-Meteo archive API
  into `raw.weather_daily`, using delete+insert per (city, date) for idempotency — proven
  safe by re-running the same date and showing row counts unchanged.
- **Transform**: dbt staging model types/cleans the raw rows; the `fct_city_daily` mart
  rounds values and adds a `had_precipitation` flag for reporting. 12 schema tests (not-null,
  accepted ranges, uniqueness) all pass.
- **Orchestration**: the same extract/load/dbt functions run here are wired into an Airflow
  DAG (`dags/weather_pipeline_dag.py`) scheduled daily, driven by the logical date so
  `airflow dags backfill` works without any hard-coded "today".
- **Result**: `marts.fct_city_daily` gives a clean, queryable daily snapshot per city —
  demonstrated above for London, Mumbai, New York, and Tokyo.